# Quant Risk Researcher API

Owns the sizing model and risk policies (decision note DEC-017). Core entry point: `backtest_portfolio` - a position-level PORTFOLIO BACKTEST before/after exposure adjustment, always paired with risk-process metrics (overfit guard).

In [ ]:
import math
from quant_api.core import DataConfig
from quant_api.risk import (
    sizing_methods, risk_policies, RiskBacktestConfig,
    backtest_portfolio, divergence_gauges, build_overlay_config,
)

print("sizing:", sizing_methods.describe())
print("policies:", risk_policies.describe())

In [ ]:
WINDOW = DataConfig(start="2026-07-15", end="2026-08-28")
n = 7712  # bars in the window
composite = [math.sin(i / 60.0) for i in range(n)]
cfg = RiskBacktestConfig(data=WINDOW)
report = backtest_portfolio(composite, data=WINDOW, config=cfg)
for side in ("before", "after"):
    p = report[side]["performance"]
    r = report[side]["risk_process"]
    print(side, "sharpe=%.2f mdd=%.2f interventions=%d tracking=%.3f" % (
        p["net_sharpe"], p["max_drawdown"], r["n_interventions"], r["mean_abs_tracking_error"]))
print("interventions:", report["interventions"][:3])

In [ ]:
expected = {"spec_ic": 0.05, "cost_model_bps": 2.294, "tracking_error_bound": 1.0}
live = {
    "score": [math.sin(i / 40.0) for i in range(1200)],
    "returns": [0.001 * math.sin(i / 40.0) for i in range(1200)],
    "fills": [{"price": 100.0, "decision_mid": 99.99, "side": 1}],
    "orders": [{}] * 4, "n_fills": 1,
    "current_positions": [1, 2, 1, 2], "target_positions": [1, 1, 1, 1],
}
print(divergence_gauges(expected, live))

In [ ]:
# Handoff artifact: validated live overlay config (consumed by live wiring).
import tempfile
payload = build_overlay_config(save=True, root=tempfile.mkdtemp(prefix="risk_state_"))
print(payload["risk"]["max_contracts"], payload["harness"]["cost_per_side"])